# Discount Leakage Analysis – Cleaning & Validation

**Dataset:** Dunnhumby "The Complete Journey" (real-world-derived)  
**Purpose:** Load, validate, clean, and export the four raw CSV files.  
Run cells top-to-bottom. Every cell has a short heading explaining what it does.

> Before running: confirm that `data/raw/` contains  
> `transaction_data.csv`, `product.csv`, `campaign_table.csv`, `campaign_desc.csv`

## 0. Imports and paths

In [11]:
import pandas as pd
import os
from pathlib import Path

# Use the project root so paths do not depend on the notebook kernel's cwd.
PROJECT_ROOT = Path(r"d:\VS Code\Python\Portfolio Project 1\discount-leakage-analysis")
RAW   = PROJECT_ROOT / "data" / "raw"
CLEAN = PROJECT_ROOT / "data" / "clean"
RPTS  = PROJECT_ROOT / "reports"

CLEAN.mkdir(parents=True, exist_ok=True)
RPTS.mkdir(parents=True, exist_ok=True)

# check files exist before loading
required = ["transaction_data.csv", "product.csv", "campaign_table.csv", "campaign_desc.csv"]
missing  = [f for f in required if not (RAW / f).exists()]

if missing:
    raise FileNotFoundError(
        f"Missing files in data/raw/: {missing}\n"
        "Download from https://www.dunnhumby.com/source-files/ and unzip into data/raw/."
    )
print(f"Project root: {PROJECT_ROOT}")
print("All required files found.")

Project root: d:\VS Code\Python\Portfolio Project 1\discount-leakage-analysis
All required files found.


## 1. Load raw files – shape, dtypes, null counts

Actual raw column names (verified from the real CSVs):

| File | Key columns |
|---|---|
| `transaction_data.csv` | `household_key, BASKET_ID, DAY, PRODUCT_ID, QUANTITY, SALES_VALUE, STORE_ID, RETAIL_DISC, TRANS_TIME, WEEK_NO, COUPON_DISC, COUPON_MATCH_DISC` |
| `product.csv` | `PRODUCT_ID, MANUFACTURER, DEPARTMENT, BRAND, COMMODITY_DESC, SUB_COMMODITY_DESC, CURR_SIZE_OF_PRODUCT` |
| `campaign_table.csv` | `DESCRIPTION, household_key, CAMPAIGN` ← DESCRIPTION here = campaign type (TypeA/B/C); we drop it |
| `campaign_desc.csv` | `DESCRIPTION, CAMPAIGN, START_DAY, END_DAY` ← DESCRIPTION here = campaign type label; we keep it |

We lowercase all column names immediately after loading.

In [12]:
# load the four raw files
tx   = pd.read_csv(os.path.join(RAW, "transaction_data.csv"))
prod = pd.read_csv(os.path.join(RAW, "product.csv"))
ct   = pd.read_csv(os.path.join(RAW, "campaign_table.csv"))
cd   = pd.read_csv(os.path.join(RAW, "campaign_desc.csv"))

# lowercase all column names so they match SQL table definitions
tx.columns   = tx.columns.str.lower()
prod.columns = prod.columns.str.lower()
ct.columns   = ct.columns.str.lower()
cd.columns   = cd.columns.str.lower()

for name, df in [("transaction_data", tx), ("product", prod),
                 ("campaign_table", ct), ("campaign_desc", cd)]:
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.dtypes)
    print("Nulls:")
    print(df.isnull().sum())


=== transaction_data ===
Shape: (2595732, 12)
Columns: ['household_key', 'basket_id', 'day', 'product_id', 'quantity', 'sales_value', 'store_id', 'retail_disc', 'trans_time', 'week_no', 'coupon_disc', 'coupon_match_disc']
household_key          int64
basket_id              int64
day                    int64
product_id             int64
quantity               int64
sales_value          float64
store_id               int64
retail_disc          float64
trans_time             int64
week_no                int64
coupon_disc          float64
coupon_match_disc    float64
dtype: object
Nulls:
household_key        0
basket_id            0
day                  0
product_id           0
quantity             0
sales_value          0
store_id             0
retail_disc          0
trans_time           0
week_no              0
coupon_disc          0
coupon_match_disc    0
dtype: int64

=== product ===
Shape: (92353, 7)
Columns: ['product_id', 'manufacturer', 'department', 'brand', 'commodity_desc', 'su

## 2. Drop duplicate rows

In [13]:
# exact duplicate rows in transactions
before = len(tx)
tx = tx.drop_duplicates()
print(f"transaction_data: dropped {before - len(tx)} exact duplicate rows")

# duplicate (household_key, campaign) pairs in campaign_table
# campaign_table columns after lowercasing: description, household_key, campaign
# 'description' here = campaign type (TypeA/B/C); we deduplicate on the key pair only
before_ct = len(ct)
ct = ct.drop_duplicates(subset=["household_key", "campaign"])
print(f"campaign_table: dropped {before_ct - len(ct)} duplicate (household_key, campaign) rows")

transaction_data: dropped 0 exact duplicate rows
campaign_table: dropped 0 duplicate (household_key, campaign) rows


## 3. Validate discount sign and suspicious rows

The Dunnhumby dictionary says `retail_disc`, `coupon_disc`, and `coupon_match_disc`  
are **negative** when a discount is applied (price reduction from the shelf price).  
We print the min/max of each to confirm this convention before we flip signs.

In [14]:
# sign check on the three raw discount columns
# after lowercasing: retail_disc, coupon_disc, coupon_match_disc
disc_cols = ["retail_disc", "coupon_disc", "coupon_match_disc"]
print("--- Discount column sign check ---")
print(tx[disc_cols].agg(["min", "max"]))

# confirm expectation: all mins should be <= 0, all maxes should be <= 0 (or tiny positive = rounding)
mostly_negative = (tx[disc_cols] <= 0).all().all()
if mostly_negative:
    print("\nSign convention confirmed: discount columns are non-positive (negative = discount applied).")
else:
    print("\nWARNING: some discount values are positive. Review before proceeding.")

# rows with suspicious quantity or sales_value
bad_qty   = (tx["quantity"] <= 0).sum()
bad_sales = (tx["sales_value"] < 0).sum()

# orphan product ids (not in product table)
orphan_ids = (~tx["product_id"].isin(prod["product_id"])).sum()

print(f"\nRows with quantity <= 0:                 {bad_qty}")
print(f"Rows with sales_value < 0:               {bad_sales}")
print(f"Rows with product_id not in product.csv: {orphan_ids}")

--- Discount column sign check ---
     retail_disc  coupon_disc  coupon_match_disc
min      -180.00       -55.93               -7.7
max         3.99         0.00                0.0


Rows with quantity <= 0:                 14466
Rows with sales_value < 0:               0
Rows with product_id not in product.csv: 0


## 4. Create derived columns

Because the raw discount columns are **negative** (confirmed in step 3),  
we negate them to get positive discount amounts that are easier to work with.

- `retailer_disc` = retailer-funded discount (retail_disc) + retailer's portion of coupon match (coupon_match_disc)  
- `mfr_disc`      = manufacturer coupon face value (coupon_disc, funded by the brand, not the retailer)  
- `gross_sales`   = what the shopper would have paid without any discount (revenue basis for discount rate)

In [15]:
# flip negative signs → positive discount amounts
tx["retailer_disc"] = -(tx["retail_disc"] + tx["coupon_match_disc"])
tx["mfr_disc"]      = -(tx["coupon_disc"])

# gross_sales = what the shopper would have paid at full price
# sales_value is the net amount paid; add back both discount types to reconstruct full price
tx["gross_sales"]   = tx["sales_value"] + tx["retailer_disc"] + tx["mfr_disc"]

# quick sanity: gross_sales should always be >= sales_value
negative_gross = (tx["gross_sales"] < 0).sum()
print(f"Rows with negative gross_sales (sanity check, expect 0): {negative_gross}")
print(tx[["sales_value", "retailer_disc", "mfr_disc", "gross_sales"]].describe().round(2))

Rows with negative gross_sales (sanity check, expect 0): 17
       sales_value  retailer_disc    mfr_disc  gross_sales
count   2595732.00     2595732.00  2595732.00   2595732.00
mean          3.10           0.54        0.02         3.66
std           4.18           1.25        0.22         4.67
min           0.00          -3.99       -0.00        -0.01
25%           1.29          -0.00       -0.00         1.59
50%           2.00           0.04       -0.00         2.59
75%           3.49           0.69        0.00         3.99
max         840.00         180.00       55.93       840.00


## 5. Drop returns / corrections

Rows with `quantity <= 0` or `sales_value < 0` are returns or price corrections.  
We drop them because they distort discount totals.  
We **keep** `sales_value == 0` rows — these are legitimately free promoted items.

In [16]:
before_rows  = len(tx)
before_money = tx["sales_value"].sum()

# flag rows to drop: returns/corrections only
drop_mask = (tx["quantity"] <= 0) | (tx["sales_value"] < 0)

dropped_rows  = drop_mask.sum()
dropped_money = tx.loc[drop_mask, "sales_value"].sum()

tx = tx[~drop_mask].copy()

pct_rows  = 100 * dropped_rows  / before_rows
pct_money = 100 * dropped_money / before_money

print(f"Rows dropped (returns/corrections): {dropped_rows} ({pct_rows:.2f}% of total)")
print(f"Sales value removed:               ${dropped_money:,.2f} ({pct_money:.2f}% of total)")

if pct_rows > 1:
    print("WARNING: >1% of rows removed — review whether this is expected for this dataset.")

print(f"\nRows remaining: {len(tx):,}")

Rows dropped (returns/corrections): 14466 (0.56% of total)
Sales value removed:               $19.28 (0.00% of total)

Rows remaining: 2,581,266


## 6. Review departments – exclude non-merchandise rows

Some departments (e.g. blank labels, coupon/misc categories) are not
real merchandise. The list below is printed so you can decide which to exclude.

**TODO:** After running this cell, review the printed department list.  
Edit `exclude_depts` to add any departments that are not genuine merchandise  
(e.g. `'COUPON'`, `'MISC ITEMS'`, fuel types). Then re-run from this cell.

In [17]:
# start with an empty exclusion list — edit after reviewing the printed table
exclude_depts = []
# example after review: exclude_depts = ["COUPON", "MISC ITEMS"]

# product.csv lowercased columns: product_id, manufacturer, department, brand,
#   commodity_desc, sub_commodity_desc, curr_size_of_product
# we only need product_id and department here
tx_with_dept = tx.merge(prod[["product_id", "department"]], on="product_id", how="left")

dept_sales = (
    tx_with_dept
    .groupby("department", dropna=False)["sales_value"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
dept_sales.columns = ["department", "total_sales_value"]
print("All departments sorted by total sales (review this list):")
print(dept_sales.to_string(index=False))

# apply exclusion
if exclude_depts:
    before_excl = len(tx)
    keep_ids = prod.loc[~prod["department"].isin(exclude_depts), "product_id"]
    tx = tx[tx["product_id"].isin(keep_ids)].copy()
    print(f"\nExcluded departments: {exclude_depts}")
    print(f"Rows removed: {before_excl - len(tx):,}")
else:
    print("\nNo departments excluded yet. Edit exclude_depts and re-run if needed.")

All departments sorted by total sales (review this list):
     department  total_sales_value
        GROCERY         4093810.84
        DRUG GM         1055357.16
        PRODUCE          557437.89
           MEAT          548786.49
      KIOSK-GAS          544222.28
     MEAT-PCKGD          412436.77
           DELI          260865.94
         PASTRY          121739.86
MISC SALES TRAN          119960.04
      NUTRITION           97669.04
  SEAFOOD-PCKGD           63093.62
         FLORAL           39653.34
      COSMETICS           32360.37
      SALAD BAR           30953.30
        SEAFOOD           29496.92
        SPIRITS           21670.88
   MISC. TRANS.           10678.69
  GARDEN CENTER            7667.82
TRAVEL & LEISUR            2550.38
    CHEF SHOPPE            2290.80
     RESTAURANT            1986.76
 COUP/STR & MFG            1027.72
 FROZEN GROCERY             653.26
     AUTOMOTIVE             452.66
             RX             159.56
   GM MERCH EXP             101.

## 7. Quick EDA – overall discount rates

In [18]:
# overall retailer discount rate
overall_rate = tx["retailer_disc"].sum() / tx["gross_sales"].sum()
print(f"Overall retailer discount rate: {overall_rate:.2%}")

# what share of line items have any retailer discount
pct_discounted = (tx["retailer_disc"] > 0).mean()
print(f"Share of line items with retailer_disc > 0: {pct_discounted:.2%}")

# household-level discount rate distribution
hh = tx.groupby("household_key").agg(
    retailer_disc=("retailer_disc", "sum"),
    gross_sales=("gross_sales", "sum")
)
hh["disc_rate"] = hh["retailer_disc"] / hh["gross_sales"]

print("\nHousehold discount rate percentiles:")
print(hh["disc_rate"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).round(4).to_string())

Overall retailer discount rate: 14.81%
Share of line items with retailer_disc > 0: 50.74%

Household discount rate percentiles:
0.10    0.0942
0.25    0.1178
0.50    0.1477
0.75    0.1814
0.90    0.2171


## 8. Control totals, save cleaned files

In [19]:
# print and save control totals
n_rows    = len(tx)
sum_sales = tx["sales_value"].sum()
sum_ret_d = tx["retailer_disc"].sum()
sum_mfr_d = tx["mfr_disc"].sum()
sum_gross = tx["gross_sales"].sum()

totals_str = (
    f"=== Control Totals (transactions_clean.csv) ===\n"
    f"Row count:          {n_rows:,}\n"
    f"Sum sales_value:    {sum_sales:,.2f}\n"
    f"Sum retailer_disc:  {sum_ret_d:,.2f}\n"
    f"Sum mfr_disc:       {sum_mfr_d:,.2f}\n"
    f"Sum gross_sales:    {sum_gross:,.2f}\n"
)
print(totals_str)

with open(os.path.join(RPTS, "control_totals.txt"), "w") as f:
    f.write(totals_str)
print("Saved reports/control_totals.txt")

=== Control Totals (transactions_clean.csv) ===
Row count:          2,581,266
Sum sales_value:    8,057,443.80
Sum retailer_disc:  1,405,913.10
Sum mfr_disc:       32,283.01
Sum gross_sales:    9,495,639.91

Saved reports/control_totals.txt


In [20]:
# ----- transactions_clean -----
# Raw tx columns after lowercasing:
#   household_key, basket_id, day, product_id, quantity, sales_value,
#   store_id, retail_disc, trans_time, week_no, coupon_disc, coupon_match_disc
#   + our new: retailer_disc, mfr_disc, gross_sales
# We keep exactly the 11 columns that match the SQL transactions table.
tx_clean = tx[[
    "household_key", "basket_id", "day", "week_no",
    "product_id", "store_id", "quantity",
    "sales_value", "retailer_disc", "mfr_disc", "gross_sales"
]].copy()

# ----- products_clean -----
# Raw prod columns after lowercasing:
#   product_id, manufacturer, department, brand, commodity_desc,
#   sub_commodity_desc, curr_size_of_product
# We keep only the 4 columns used in SQL.
prod_clean = prod[["product_id", "department", "brand", "commodity_desc"]].copy()

# ----- campaign_table_clean -----
# Raw ct columns after lowercasing: description, household_key, campaign
# 'description' here = campaign type label — NOT needed in this table (it lives in campaign_desc).
# We keep only the 2 key columns.
ct_clean = ct[["household_key", "campaign"]].copy()

# ----- campaign_desc_clean -----
# Raw cd columns after lowercasing: description, campaign, start_day, end_day
# 'description' IS the campaign type label (TypeA / TypeB / TypeC).
# SQL campaign_desc table expects columns: campaign, description, start_day, end_day
cd_clean = cd[["campaign", "description", "start_day", "end_day"]].copy()

# save
tx_clean.to_csv(os.path.join(CLEAN, "transactions_clean.csv"), index=False)
prod_clean.to_csv(os.path.join(CLEAN, "products_clean.csv"), index=False)
ct_clean.to_csv(os.path.join(CLEAN, "campaign_table_clean.csv"), index=False)
cd_clean.to_csv(os.path.join(CLEAN, "campaign_desc_clean.csv"), index=False)

print("Saved to data/clean/:")
print(f"  transactions_clean.csv   {len(tx_clean):,} rows  cols: {list(tx_clean.columns)}")
print(f"  products_clean.csv       {len(prod_clean):,} rows  cols: {list(prod_clean.columns)}")
print(f"  campaign_table_clean.csv {len(ct_clean):,} rows  cols: {list(ct_clean.columns)}")
print(f"  campaign_desc_clean.csv  {len(cd_clean):,} rows  cols: {list(cd_clean.columns)}")
print("\nAll column names match SQL table definitions.")

Saved to data/clean/:
  transactions_clean.csv   2,581,266 rows  cols: ['household_key', 'basket_id', 'day', 'week_no', 'product_id', 'store_id', 'quantity', 'sales_value', 'retailer_disc', 'mfr_disc', 'gross_sales']
  products_clean.csv       92,353 rows  cols: ['product_id', 'department', 'brand', 'commodity_desc']
  campaign_table_clean.csv 7,208 rows  cols: ['household_key', 'campaign']
  campaign_desc_clean.csv  30 rows  cols: ['campaign', 'description', 'start_day', 'end_day']

All column names match SQL table definitions.
